# 01 - MovieLens-1M Data Audit

Locate or explicitly acquire the official MovieLens-1M release, parse its raw
files without model-specific transformations, validate it against GroupLens
documentation, and prepare a reproducible handoff for the cold-start protocol.

## Notebook Linkage and Boundaries

Notebook 00 defines the environment-variable and artifact-root contract used
here. This notebook repeats the minimal path discovery so it remains
independently executable; it does not rely on notebook 00's in-memory state.

**Input:** An official `ml-1m.zip` archive or extracted `ml-1m` directory,
supplied locally or attached as a Kaggle dataset.

**Output:** Lossless canonical tables, source hashes, checks, feature inventory,
and an audit manifest under the hidden artifact workspace.

Notebook 02 must verify this manifest before deriving labels, contiguous IDs,
item cohorts, candidates, splits, or Cold/Warm-Up A/B/C phases. None of those
protocol decisions belong here.

## Runtime and Path Contract

The path overrides are inherited from notebook 00:
`COLDSTART_PROJECT_ROOT`, `COLDSTART_INPUT_ROOT`,
`COLDSTART_WORKSPACE_ROOT`, and `COLDSTART_ARTIFACT_ROOT`.

In [21]:
from __future__ import annotations

import hashlib
import importlib
import json
import os
import platform
import shutil
import sys
import urllib.request
import uuid
import zipfile
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib import metadata, util
from pathlib import Path
from typing import Any, BinaryIO, Iterable, Iterator

from IPython.display import Markdown, display


def show_records(
    records: Iterable[dict[str, Any]], columns: list[str] | None = None
) -> None:
    rows = list(records)
    pandas = safe_import("pandas")
    if pandas is None or not rows:
        display(rows)
        return
    table = pandas.DataFrame(rows)
    display(table if columns is None else table.reindex(columns=columns))


def safe_import(module: str) -> Any | None:
    try:
        return importlib.import_module(module)
    except Exception:
        return None


def enabled(variable: str, default: bool = False) -> bool:
    raw = os.environ.get(variable)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}

In [22]:
def detect_execution_context() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def discover_project_root(start: Path) -> tuple[Path, str]:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve(), "COLDSTART_PROJECT_ROOT"

    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve(), ".git marker"
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").exists():
            return candidate.resolve(), "repository markers"
    return start.resolve(), "working-directory fallback"


def configured_path(variable: str, default: Path, base: Path) -> Path:
    value = Path(os.environ.get(variable, str(default))).expanduser()
    return (base / value).resolve() if not value.is_absolute() else value.resolve()

In [23]:
EXECUTION_CONTEXT = detect_execution_context()
CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT, PROJECT_ROOT_SOURCE = discover_project_root(CURRENT_WORKING_DIRECTORY)

if EXECUTION_CONTEXT == "kaggle":
    default_workspace = Path("/kaggle/working")
    default_input = Path("/kaggle/input")
elif EXECUTION_CONTEXT == "colab":
    default_workspace = Path("/content")
    default_input = Path("/content/data")
else:
    default_workspace = PROJECT_ROOT / ".notebook"
    default_input = PROJECT_ROOT / "data"

WORKSPACE_ROOT = configured_path("COLDSTART_WORKSPACE_ROOT", default_workspace, PROJECT_ROOT)
INPUT_ROOT = configured_path("COLDSTART_INPUT_ROOT", default_input, PROJECT_ROOT)
ARTIFACT_ROOT = configured_path(
    "COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts", PROJECT_ROOT
)
CACHE_ROOT = WORKSPACE_ROOT / "cache" / "ml-1m"
AUDIT_OUTPUT_ROOT = ARTIFACT_ROOT / "processed" / "ml-1m" / "audit-v2"
NOTEBOOK_00_PATH = PROJECT_ROOT / "notebooks" / "00_environment_and_reproducibility.ipynb"

os.environ["MPLCONFIGDIR"] = str(WORKSPACE_ROOT / "cache" / "matplotlib")

PATH_CONTRACT = {
    "execution_context": EXECUTION_CONTEXT,
    "python_version": platform.python_version(),
    "project_root": str(PROJECT_ROOT),
    "project_root_source": PROJECT_ROOT_SOURCE,
    "input_root": str(INPUT_ROOT),
    "workspace_root": str(WORKSPACE_ROOT),
    "artifact_root": str(ARTIFACT_ROOT),
    "audit_output_root": str(AUDIT_OUTPUT_ROOT),
    "notebook_00_visible": NOTEBOOK_00_PATH.is_file(),
}

show_records([PATH_CONTRACT])

,execution_context,python_version,project_root,project_root_source,input_root,workspace_root,artifact_root,audit_output_root,notebook_00_visible
0,local,3.12.3,/workspace/HungPH/coldstart-recsys,.git marker,/workspace/HungPH/coldstart-recsys/data,/workspace/HungPH/coldstart-recsys/.notebook,/workspace/HungPH/coldstart-recsys/.notebook/a...,/workspace/HungPH/coldstart-recsys/.notebook/a...,True


## Dependency Contract

Pandas and NumPy are required for parsing and validation. Matplotlib is
optional for the diagnostic figure only. No package is installed by this
notebook, and no network access is required for attached data.

In [24]:
DEPENDENCY_SPECS = (
    ("numpy", "numpy", True, "array and numeric support"),
    ("pandas", "pandas", True, "lossless parsing and tabular audit"),
    ("matplotlib", "matplotlib", False, "diagnostic plots"),
)

DEPENDENCY_AUDIT = []
for module_name, distribution, required, purpose in DEPENDENCY_SPECS:
    module = safe_import(module_name)
    try:
        version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        version = None
    DEPENDENCY_AUDIT.append(
        {
            "distribution": distribution,
            "required": required,
            "importable": module is not None,
            "version": version,
            "purpose": purpose,
            "status": "PASS" if module is not None else ("FAIL" if required else "WARN"),
        }
    )

MISSING_REQUIRED = [
    row["distribution"]
    for row in DEPENDENCY_AUDIT
    if row["required"] and not row["importable"]
]
pandas = safe_import("pandas")
numpy = safe_import("numpy")
pyplot = safe_import("matplotlib.pyplot")

show_records(
    DEPENDENCY_AUDIT,
    ["distribution", "required", "importable", "version", "status", "purpose"],
)

,distribution,required,importable,version,status,purpose
0,numpy,True,True,2.4.4,PASS,array and numeric support
1,pandas,True,True,3.0.3,PASS,lossless parsing and tabular audit
2,matplotlib,False,False,NaN,WARN,diagnostic plots


## Official MovieLens-1M Contract

Schema and value constraints below come from the GroupLens MovieLens-1M
README. The release page describes this as a stable benchmark released in
February 2003. The archive license permits research use with attribution and
restricts redistribution; retain the README with every audited input.

- Release page: <https://grouplens.org/datasets/movielens/1m/>
- README: <https://files.grouplens.org/datasets/movielens/ml-1m-README.txt>
- Archive: <https://files.grouplens.org/datasets/movielens/ml-1m.zip>

In [25]:
DATASET_NAME = "MovieLens-1M"
DATASET_RELEASE = "ml-1m"
AUDIT_SCHEMA_VERSION = "ml1m-audit-v2"
OFFICIAL_ARCHIVE_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
OFFICIAL_README_URL = "https://files.grouplens.org/datasets/movielens/ml-1m-README.txt"
OFFICIAL_ARCHIVE_MD5 = "c4d9eecfca2ab87c1945afe126590906"
TRUSTED_ARCHIVE_SHA256 = "a6898adb50b9ca05aa231689da44c217cb524e7ebd39d264c56e2832f2c54e20"
TRUSTED_ARCHIVE_SIZE = 5_917_549

TRUSTED_MEMBER_FINGERPRINTS = {
    "ratings.dat": {
        "size_bytes": 24_594_131,
        "sha256": "506d64ca44484487c11dc2d9a28de5c54948213e6b96285e298afe28d6ea4e0f",
    },
    "users.dat": {
        "size_bytes": 134_368,
        "sha256": "1dc3a95300cb19f7c10e027daf29b8cdf1d908dab65b5edfb45950aa389a10a3",
    },
    "movies.dat": {
        "size_bytes": 171_308,
        "sha256": "0140fc2356357c1a851d0f52e893a1e4d3696df632c4141cea8d5bc3d621f0b9",
    },
    "README": {
        "size_bytes": 5_577,
        "sha256": "2d5ddce86ccbd247207456165827b4b9ddf27282bca93807d4c763e28414f133",
    },
}

SOURCE_SCHEMAS = {
    "ratings.dat": {
        "source_columns": ["UserID", "MovieID", "Rating", "Timestamp"],
        "canonical_columns": ["user_id", "item_id", "rating", "timestamp"],
        "delimiter": "::",
        "encoding": "ISO-8859-1",
    },
    "users.dat": {
        "source_columns": ["UserID", "Gender", "Age", "Occupation", "Zip-code"],
        "canonical_columns": ["user_id", "gender", "age", "occupation", "zip_code"],
        "delimiter": "::",
        "encoding": "ISO-8859-1",
    },
    "movies.dat": {
        "source_columns": ["MovieID", "Title", "Genres"],
        "canonical_columns": ["item_id", "title", "genres"],
        "delimiter": "::",
        "encoding": "ISO-8859-1",
    },
}

CANONICAL_OUTPUT_SCHEMAS = {
    "interactions": {
        "columns": ["user_id", "item_id", "rating", "timestamp"],
        "read_csv_dtypes": {
            "user_id": "int64",
            "item_id": "int64",
            "rating": "int64",
            "timestamp": "int64",
        },
    },
    "users": {
        "columns": ["user_id", "gender", "age", "occupation", "zip_code"],
        "read_csv_dtypes": {
            "user_id": "int64",
            "gender": "string",
            "age": "int64",
            "occupation": "int64",
            "zip_code": "string",
        },
    },
    "items": {
        "columns": [
            "item_id",
            "title",
            "genres",
            "release_year",
            "release_year_parse_status",
        ],
        "read_csv_dtypes": {
            "item_id": "int64",
            "title": "string",
            "genres": "string",
            "release_year": "Int64",
            "release_year_parse_status": "string",
        },
    },
}

EXPECTED_ROWS = {"ratings": 1_000_209, "users": 6_040, "items": 3_883}
EXPECTED_USER_ID_RANGE = (1, 6_040)
EXPECTED_ITEM_ID_RANGE = (1, 3_952)
EXPECTED_RATINGS = {1, 2, 3, 4, 5}
EXPECTED_AGE_CODES = {1, 18, 25, 35, 45, 50, 56}
EXPECTED_OCCUPATION_CODES = set(range(21))
EXPECTED_GENDERS = {"F", "M"}
EXPECTED_GENRES = {
    "Action",
    "Adventure",
    "Animation",
    "Children's",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western",
}
README_REQUIRED_SNIPPETS = (
    "UserID::MovieID::Rating::Timestamp",
    "UserID::Gender::Age::Occupation::Zip-code",
    "MovieID::Title::Genres",
    "Ratings are made on a 5-star scale",
    "Each user has at least 20 ratings",
)

show_records(
    [
        {
            "file": file_name,
            "source_columns": contract["source_columns"],
            "canonical_columns": contract["canonical_columns"],
            "delimiter": contract["delimiter"],
            "encoding": contract["encoding"],
        }
        for file_name, contract in SOURCE_SCHEMAS.items()
    ]
)
show_records(
    [
        {
            "table": table_name,
            "columns": contract["columns"],
            "read_csv_dtypes": contract["read_csv_dtypes"],
        }
        for table_name, contract in CANONICAL_OUTPUT_SCHEMAS.items()
    ]
)

,file,source_columns,canonical_columns,delimiter,encoding
0,ratings.dat,"[UserID, MovieID, Rating, Timestamp]","[user_id, item_id, rating, timestamp]",::,ISO-8859-1
1,users.dat,"[UserID, Gender, Age, Occupation, Zip-code]","[user_id, gender, age, occupation, zip_code]",::,ISO-8859-1
2,movies.dat,"[MovieID, Title, Genres]","[item_id, title, genres]",::,ISO-8859-1


,table,columns,read_csv_dtypes
0,interactions,"[user_id, item_id, rating, timestamp]","{'user_id': 'int64', 'item_id': 'int64', 'rati..."
1,users,"[user_id, gender, age, occupation, zip_code]","{'user_id': 'int64', 'gender': 'string', 'age'..."
2,items,"[item_id, title, genres, release_year, release...","{'item_id': 'int64', 'title': 'string', 'genre..."


## Data Access

Resolution order is an explicit `MOVIELENS_1M_PATH`, the hidden download cache,
and then shallow candidates under the input root. `COLDSTART_ALLOW_DOWNLOAD=1`
explicitly permits an official download when no source is found. The default
remains offline and is compatible with private Kaggle kernels without internet.

In [26]:
LOGICAL_MEMBER_NAMES = {
    "ratings.dat": ("ratings.dat",),
    "users.dat": ("users.dat",),
    "movies.dat": ("movies.dat",),
    "README": ("README", "README.txt", "ml-1m-README.txt"),
}


@dataclass(frozen=True)
class DatasetSource:
    kind: str
    location: Path
    members: dict[str, str]
    origin: str


def match_members(names: Iterable[str]) -> dict[str, str] | None:
    clean_names = [name for name in names if "__MACOSX" not in name]
    matches: dict[str, str] = {}
    for logical_name, accepted_names in LOGICAL_MEMBER_NAMES.items():
        candidates = [name for name in clean_names if Path(name).name in accepted_names]
        if len(candidates) != 1:
            return None
        matches[logical_name] = candidates[0]
    return matches


def inspect_candidate(path: Path, origin: str) -> DatasetSource | None:
    path = path.expanduser().resolve()
    if path.is_file() and zipfile.is_zipfile(path):
        try:
            trusted_archive = (
                path.stat().st_size == TRUSTED_ARCHIVE_SIZE
                and file_digest(path, "md5") == OFFICIAL_ARCHIVE_MD5
                and file_digest(path, "sha256") == TRUSTED_ARCHIVE_SHA256
            )
        except OSError:
            return None
        if not trusted_archive:
            return None
        with zipfile.ZipFile(path) as archive:
            members = match_members(archive.namelist())
        return DatasetSource("zip", path, members, origin) if members else None

    if path.is_dir():
        for root in (path, path / "ml-1m"):
            if not root.is_dir():
                continue
            names = [entry.name for entry in root.iterdir() if entry.is_file()]
            members = match_members(names)
            if members:
                try:
                    trusted = all(
                        (root / members[logical_name]).stat().st_size
                        == TRUSTED_MEMBER_FINGERPRINTS[logical_name]["size_bytes"]
                        and file_digest(root / members[logical_name], "sha256")
                        == TRUSTED_MEMBER_FINGERPRINTS[logical_name]["sha256"]
                        for logical_name in LOGICAL_MEMBER_NAMES
                    )
                except OSError:
                    trusted = False
                if trusted:
                    return DatasetSource("directory", root, members, origin)
    return None

In [27]:
def candidate_paths() -> list[tuple[Path, str]]:
    """Return plausible MovieLens-1M sources in deterministic priority order.

    Kaggle can mount attached datasets several directory levels below
    /kaggle/input, for example:

        /kaggle/input/datasets/<owner>/<dataset-slug>/ml-1m/

    Therefore direct-child scanning alone is not sufficient.
    """
    candidates: list[tuple[Path, str]] = []

    # Highest priority: an explicit user-provided path.
    explicit = os.environ.get("MOVIELENS_1M_PATH")
    if explicit:
        candidates.append((Path(explicit), "MOVIELENS_1M_PATH"))

    # Stable conventional locations used by local, Colab, and Kaggle runs.
    candidates.extend(
        [
            (CACHE_ROOT / "downloads" / "ml-1m.zip", "hidden download cache"),
            (INPUT_ROOT / "ml-1m.zip", "input root"),
            (INPUT_ROOT / "ml-1m", "input root"),
            (INPUT_ROOT, "input root"),
        ]
    )

    if INPUT_ROOT.is_dir():
        # Preserve the original shallow Kaggle/input search.
        try:
            for child in sorted(INPUT_ROOT.iterdir()):
                candidates.extend(
                    [
                        (child, "attached input"),
                        (child / "ml-1m", "attached input"),
                        (child / "ml-1m.zip", "attached input"),
                    ]
                )
        except OSError:
            pass

        # Kaggle may mount sources under nested owner/dataset directories.
        # Discover official archives wherever they are mounted.
        try:
            for archive in sorted(INPUT_ROOT.rglob("ml-1m.zip")):
                if archive.is_file():
                    candidates.append((archive, "recursive attached input"))
        except OSError:
            pass

        # A directory containing ratings.dat is a plausible extracted
        # MovieLens-1M root. inspect_candidate() will still enforce all four
        # required files and the trusted byte-level fingerprints.
        try:
            dataset_directories = {
                ratings_path.parent
                for ratings_path in INPUT_ROOT.rglob("ratings.dat")
                if ratings_path.is_file()
            }
            for directory in sorted(
                dataset_directories,
                key=lambda path: (
                    path.name != "ml-1m",
                    len(path.parts),
                    str(path),
                ),
            ):
                candidates.append((directory, "recursive attached input"))
        except OSError:
            pass

    # Remove duplicate paths while retaining the first/highest-priority origin.
    unique: list[tuple[Path, str]] = []
    seen: set[str] = set()
    for path, origin in candidates:
        key = str(path.expanduser().resolve())
        if key not in seen:
            seen.add(key)
            unique.append((path, origin))
    return unique


def file_digest(path: Path, algorithm: str = "sha256") -> str:
    digest = hashlib.new(algorithm)
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_official_archive(destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".part")
    request = urllib.request.Request(
        OFFICIAL_ARCHIVE_URL,
        headers={"User-Agent": "coldstart-recsys-data-audit/1.0"},
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with temporary.open("wb") as output:
                shutil.copyfileobj(response, output)

        observed_size = temporary.stat().st_size
        observed_md5 = file_digest(temporary, "md5")
        observed_sha256 = file_digest(temporary, "sha256")
        if (
            observed_size != TRUSTED_ARCHIVE_SIZE
            or observed_md5 != OFFICIAL_ARCHIVE_MD5
            or observed_sha256 != TRUSTED_ARCHIVE_SHA256
        ):
            raise ValueError(
                "Official archive fingerprint mismatch: "
                f"size={observed_size}, md5={observed_md5}, "
                f"sha256={observed_sha256}"
            )
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


In [28]:
DOWNLOAD_ALLOWED = enabled("COLDSTART_ALLOW_DOWNLOAD", default=False)
SOURCE_MESSAGES: list[str] = []
DATA_SOURCE: DatasetSource | None = None

SOURCE_CANDIDATES = candidate_paths()
EXISTING_SOURCE_CANDIDATES: list[str] = []

for source_path, source_origin in SOURCE_CANDIDATES:
    try:
        if source_path.expanduser().exists():
            EXISTING_SOURCE_CANDIDATES.append(
                str(source_path.expanduser().resolve())
            )
    except OSError:
        pass

    candidate = inspect_candidate(source_path, source_origin)
    if candidate is not None:
        DATA_SOURCE = candidate
        break

if DATA_SOURCE is None and DOWNLOAD_ALLOWED:
    try:
        downloaded = download_official_archive(
            CACHE_ROOT / "downloads" / "ml-1m.zip"
        )
        DATA_SOURCE = inspect_candidate(
            downloaded,
            "explicit official download",
        )
    except Exception as error:
        SOURCE_MESSAGES.append(f"Download failed: {error!r}")

if DATA_SOURCE is None:
    if EXISTING_SOURCE_CANDIDATES:
        SOURCE_MESSAGES.append(
            "MovieLens-like paths were discovered, but none passed the "
            "required file, size, and SHA-256 checks. Inspect the candidate "
            "paths below or set MOVIELENS_1M_PATH explicitly."
        )
    else:
        SOURCE_MESSAGES.append(
            "MovieLens-1M was not found. Attach ml-1m.zip, set "
            "MOVIELENS_1M_PATH, or explicitly set "
            "COLDSTART_ALLOW_DOWNLOAD=1 in an internet-enabled runtime."
        )

show_records(
    [
        {
            "source_found": DATA_SOURCE is not None,
            "source_kind": DATA_SOURCE.kind if DATA_SOURCE else None,
            "source_location": str(DATA_SOURCE.location) if DATA_SOURCE else None,
            "source_origin": DATA_SOURCE.origin if DATA_SOURCE else None,
            "download_allowed": DOWNLOAD_ALLOWED,
            "candidate_count": len(SOURCE_CANDIDATES),
            "existing_candidates_checked": EXISTING_SOURCE_CANDIDATES,
            "messages": SOURCE_MESSAGES,
        }
    ]
)


,source_found,source_kind,source_location,source_origin,download_allowed,candidate_count,existing_candidates_checked,messages
0,True,zip,/workspace/HungPH/coldstart-recsys/data/ml-1m.zip,input root,False,6,[/workspace/HungPH/coldstart-recsys/data/ml-1m...,[]


## Source Inventory and Documentation Validation

ZIP size and whole-archive MD5/SHA-256 are checked before any member is
decompressed. Every member must then match the size and SHA-256 derived from the
MD5-verified official release. This also authenticates extracted directories.

In [29]:
@contextmanager
def open_source_member(source: DatasetSource, logical_name: str) -> Iterator[BinaryIO]:
    member_name = source.members[logical_name]
    if source.kind == "directory":
        with (source.location / member_name).open("rb") as stream:
            yield stream
        return

    with zipfile.ZipFile(source.location) as archive:
        with archive.open(member_name, "r") as stream:
            yield stream


def stream_hash(stream: BinaryIO, algorithm: str = "sha256") -> str:
    digest = hashlib.new(algorithm)
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
    return digest.hexdigest()


def source_member_size(source: DatasetSource, logical_name: str) -> int:
    member_name = source.members[logical_name]
    if source.kind == "directory":
        return (source.location / member_name).stat().st_size
    with zipfile.ZipFile(source.location) as archive:
        return archive.getinfo(member_name).file_size


SOURCE_INVENTORY: list[dict[str, Any]] = []
README_TEXT = ""
ARCHIVE_SHA256 = None
ARCHIVE_MD5 = None
ARCHIVE_SIZE = None
ARCHIVE_FINGERPRINT_VALID = False
MEMBER_FINGERPRINT_VALID = False

if DATA_SOURCE is not None:
    if DATA_SOURCE.kind == "zip":
        ARCHIVE_SIZE = DATA_SOURCE.location.stat().st_size
        if ARCHIVE_SIZE == TRUSTED_ARCHIVE_SIZE:
            ARCHIVE_SHA256 = file_digest(DATA_SOURCE.location, "sha256")
            ARCHIVE_MD5 = file_digest(DATA_SOURCE.location, "md5")
        ARCHIVE_FINGERPRINT_VALID = (
            ARCHIVE_SIZE == TRUSTED_ARCHIVE_SIZE
            and ARCHIVE_MD5 == OFFICIAL_ARCHIVE_MD5
            and ARCHIVE_SHA256 == TRUSTED_ARCHIVE_SHA256
        )
    else:
        ARCHIVE_FINGERPRINT_VALID = True

    if ARCHIVE_FINGERPRINT_VALID:
        for logical_name, member_name in DATA_SOURCE.members.items():
            expected = TRUSTED_MEMBER_FINGERPRINTS[logical_name]
            member_size = source_member_size(DATA_SOURCE, logical_name)
            member_sha256 = None
            if member_size == expected["size_bytes"]:
                with open_source_member(DATA_SOURCE, logical_name) as stream:
                    member_sha256 = stream_hash(stream)
            SOURCE_INVENTORY.append(
                {
                    "logical_name": logical_name,
                    "source_member": member_name,
                    "size_bytes": member_size,
                    "expected_size_bytes": expected["size_bytes"],
                    "sha256": member_sha256,
                    "expected_sha256": expected["sha256"],
                    "matches_trusted_fingerprint": (
                        member_size == expected["size_bytes"]
                        and member_sha256 == expected["sha256"]
                    ),
                }
            )

        MEMBER_FINGERPRINT_VALID = (
            len(SOURCE_INVENTORY) == len(TRUSTED_MEMBER_FINGERPRINTS)
            and all(row["matches_trusted_fingerprint"] for row in SOURCE_INVENTORY)
        )
        if MEMBER_FINGERPRINT_VALID:
            with open_source_member(DATA_SOURCE, "README") as stream:
                README_TEXT = stream.read().decode("ISO-8859-1")

README_AUTHORITY_VALID = bool(README_TEXT) and all(
    snippet in README_TEXT for snippet in README_REQUIRED_SNIPPETS
)
SOURCE_IDENTITY_VALID = (
    DATA_SOURCE is not None and ARCHIVE_FINGERPRINT_VALID and MEMBER_FINGERPRINT_VALID
)
ARCHIVE_IDENTITY_STATUS = "PASS" if SOURCE_IDENTITY_VALID else "FAIL"

show_records(
    SOURCE_INVENTORY,
    [
        "logical_name",
        "source_member",
        "size_bytes",
        "expected_size_bytes",
        "sha256",
        "matches_trusted_fingerprint",
    ],
)
show_records(
    [
        {
            "readme_contract_valid": README_AUTHORITY_VALID,
            "archive_identity_status": ARCHIVE_IDENTITY_STATUS,
            "archive_md5": ARCHIVE_MD5,
            "expected_archive_md5": OFFICIAL_ARCHIVE_MD5,
            "archive_sha256": ARCHIVE_SHA256,
            "expected_archive_sha256": TRUSTED_ARCHIVE_SHA256,
            "archive_size": ARCHIVE_SIZE,
            "member_fingerprints_valid": MEMBER_FINGERPRINT_VALID,
        }
    ]
)

,logical_name,source_member,size_bytes,expected_size_bytes,sha256,matches_trusted_fingerprint
0,ratings.dat,ml-1m/ratings.dat,24594131,24594131,506d64ca44484487c11dc2d9a28de5c54948213e6b9628...,True
1,users.dat,ml-1m/users.dat,134368,134368,1dc3a95300cb19f7c10e027daf29b8cdf1d908dab65b5e...,True
2,movies.dat,ml-1m/movies.dat,171308,171308,0140fc2356357c1a851d0f52e893a1e4d3696df632c414...,True
3,README,ml-1m/README,5577,5577,2d5ddce86ccbd247207456165827b4b9ddf27282bca938...,True


,readme_contract_valid,archive_identity_status,archive_md5,expected_archive_md5,archive_sha256,expected_archive_sha256,archive_size,member_fingerprints_valid
0,True,PASS,c4d9eecfca2ab87c1945afe126590906,c4d9eecfca2ab87c1945afe126590906,a6898adb50b9ca05aa231689da44c217cb524e7ebd39d2...,a6898adb50b9ca05aa231689da44c217cb524e7ebd39d2...,5917549,True


## Lossless Parsing

Canonical names use `item_id` consistently with the project, while the manifest
preserves the official source names. Ratings and timestamps remain unchanged.
User demographic codes remain codes, ZIP codes remain strings, and movie title
and pipe-delimited genres remain intact.

In [30]:
def read_source_table(
    source: DatasetSource,
    logical_name: str,
    columns: list[str],
    dtypes: dict[str, str],
) -> Any:
    if pandas is None:
        raise RuntimeError("pandas is required to parse MovieLens-1M")
    with open_source_member(source, logical_name) as stream:
        return pandas.read_csv(
            stream,
            sep="::",
            engine="python",
            names=columns,
            header=None,
            dtype=dtypes,
            encoding="ISO-8859-1",
            keep_default_na=False,
            na_filter=False,
            on_bad_lines="error",
        )


def parse_movielens_1m(source: DatasetSource) -> tuple[Any, Any, Any]:
    ratings = read_source_table(
        source,
        "ratings.dat",
        ["user_id", "item_id", "rating", "timestamp"],
        {"user_id": "int64", "item_id": "int64", "rating": "int64", "timestamp": "int64"},
    )
    users = read_source_table(
        source,
        "users.dat",
        ["user_id", "gender", "age", "occupation", "zip_code"],
        {
            "user_id": "int64",
            "gender": "string",
            "age": "int64",
            "occupation": "int64",
            "zip_code": "string",
        },
    )
    items = read_source_table(
        source,
        "movies.dat",
        ["item_id", "title", "genres"],
        {"item_id": "int64", "title": "string", "genres": "string"},
    )
    return ratings, users, items

In [31]:
RATINGS = None
USERS = None
ITEMS = None
CANONICAL_ITEMS = None
PARSE_ERROR = None

if SOURCE_IDENTITY_VALID and pandas is not None:
    try:
        RATINGS, USERS, ITEMS = parse_movielens_1m(DATA_SOURCE)
        CANONICAL_ITEMS = ITEMS.copy()
        extracted_year = CANONICAL_ITEMS["title"].str.extract(
            r"\((\d{4})\)\s*$", expand=False
        )
        CANONICAL_ITEMS["release_year"] = pandas.to_numeric(
            extracted_year, errors="coerce"
        ).astype("Int64")
        CANONICAL_ITEMS["release_year_parse_status"] = CANONICAL_ITEMS[
            "release_year"
        ].notna().map({True: "parsed", False: "missing"})
    except Exception as error:
        PARSE_ERROR = repr(error)

DATA_PARSED = all(table is not None for table in (RATINGS, USERS, CANONICAL_ITEMS))

show_records(
    [
        {
            "data_parsed": DATA_PARSED,
            "parse_error": PARSE_ERROR,
            "ratings_rows": len(RATINGS) if RATINGS is not None else None,
            "users_rows": len(USERS) if USERS is not None else None,
            "items_rows": len(CANONICAL_ITEMS) if CANONICAL_ITEMS is not None else None,
        }
    ]
)

,data_parsed,parse_error,ratings_rows,users_rows,items_rows
0,True,None,1000209,6040,3883


In [32]:
SCHEMA_REPORT: list[dict[str, Any]] = []
if DATA_PARSED:
    for table_name, table in (
        ("interactions", RATINGS),
        ("users", USERS),
        ("items", CANONICAL_ITEMS),
    ):
        for column in table.columns:
            SCHEMA_REPORT.append(
                {
                    "table": table_name,
                    "column": column,
                    "dtype": str(table[column].dtype),
                    "missing": int(table[column].isna().sum()),
                    "unique": int(table[column].nunique(dropna=True)),
                }
            )

show_records(SCHEMA_REPORT, ["table", "column", "dtype", "missing", "unique"])

,table,column,dtype,missing,unique
0,interactions,user_id,int64,0,6040
1,interactions,item_id,int64,0,3706
2,interactions,rating,int64,0,5
3,interactions,timestamp,int64,0,458455
4,users,user_id,int64,0,6040
5,users,gender,string,0,2
6,users,age,int64,0,7
7,users,occupation,int64,0,21
8,users,zip_code,string,0,3439
9,items,item_id,int64,0,3883


## Integrity Checks

Critical failures block export. Warnings identify limitations that preserve
correctness, such as auditing an extracted directory rather than the original
ZIP or failing to parse a release year from an otherwise valid title.

In [33]:
AUDIT_CHECKS: list[dict[str, Any]] = []
SOURCE_REFERENCE = (
    {
        "kind": DATA_SOURCE.kind,
        "origin": DATA_SOURCE.origin,
        "name": DATA_SOURCE.location.name,
    }
    if DATA_SOURCE is not None
    else None
)


def add_check(
    category: str,
    check: str,
    passed: bool,
    observed: Any,
    expected: Any,
    failure_status: str = "FAIL",
) -> None:
    AUDIT_CHECKS.append(
        {
            "category": category,
            "check": check,
            "status": "PASS" if passed else failure_status,
            "observed": observed,
            "expected": expected,
        }
    )


add_check("environment", "required dependencies", not MISSING_REQUIRED, MISSING_REQUIRED, [])
add_check(
    "source",
    "MovieLens-1M source found",
    DATA_SOURCE is not None,
    SOURCE_REFERENCE,
    "official source",
)
add_check(
    "source",
    "required source members",
    DATA_SOURCE is not None and len(SOURCE_INVENTORY) == len(LOGICAL_MEMBER_NAMES),
    [row["logical_name"] for row in SOURCE_INVENTORY],
    list(LOGICAL_MEMBER_NAMES),
)
add_check("source", "README schema authority", README_AUTHORITY_VALID, README_AUTHORITY_VALID, True)
add_check(
    "source",
    "official archive identity",
    SOURCE_IDENTITY_VALID,
    (
        {"archive_md5": ARCHIVE_MD5, "members_valid": MEMBER_FINGERPRINT_VALID}
        if DATA_SOURCE is not None
        else "source unavailable"
    ),
    "trusted archive or exact trusted member fingerprints",
)
add_check("parsing", "all canonical tables parsed", DATA_PARSED, PARSE_ERROR, "no error")

In [34]:
TIMESTAMP_UTC = None
ITEM_ACTIVITY = None
USER_ACTIVITY = None
UNKNOWN_GENRES: set[str] = set()

if DATA_PARSED:
    TIMESTAMP_UTC = pandas.to_datetime(
        RATINGS["timestamp"], unit="s", utc=True, errors="coerce"
    )
    ITEM_ACTIVITY = RATINGS.groupby("item_id", sort=True).size()
    USER_ACTIVITY = RATINGS.groupby("user_id", sort=True).size().reindex(
        USERS["user_id"], fill_value=0
    )
    observed_genres = set(CANONICAL_ITEMS["genres"].str.split("|").explode())
    UNKNOWN_GENRES = observed_genres - EXPECTED_GENRES

    for table_name, table, expected_rows in (
        ("ratings", RATINGS, EXPECTED_ROWS["ratings"]),
        ("users", USERS, EXPECTED_ROWS["users"]),
        ("items", CANONICAL_ITEMS, EXPECTED_ROWS["items"]),
    ):
        add_check("release fingerprint", f"{table_name} row count", len(table) == expected_rows, len(table), expected_rows)
        add_check("missingness", f"{table_name} missing values", int(table.isna().sum().sum()) == 0, int(table.isna().sum().sum()), 0)

    add_check("keys", "unique user primary key", not USERS["user_id"].duplicated().any(), int(USERS["user_id"].duplicated().sum()), 0)
    add_check("keys", "unique item primary key", not CANONICAL_ITEMS["item_id"].duplicated().any(), int(CANONICAL_ITEMS["item_id"].duplicated().sum()), 0)
    duplicate_pairs = int(RATINGS.duplicated(["user_id", "item_id"]).sum())
    add_check("keys", "unique user-item ratings", duplicate_pairs == 0, duplicate_pairs, 0)

    add_check("ranges", "user ID range", USERS["user_id"].between(*EXPECTED_USER_ID_RANGE).all(), (int(USERS["user_id"].min()), int(USERS["user_id"].max())), EXPECTED_USER_ID_RANGE)
    add_check("ranges", "item ID range", CANONICAL_ITEMS["item_id"].between(*EXPECTED_ITEM_ID_RANGE).all(), (int(CANONICAL_ITEMS["item_id"].min()), int(CANONICAL_ITEMS["item_id"].max())), EXPECTED_ITEM_ID_RANGE)
    add_check("ranges", "whole-star rating values", set(RATINGS["rating"].unique()) <= EXPECTED_RATINGS, sorted(RATINGS["rating"].unique().tolist()), sorted(EXPECTED_RATINGS))
    add_check("ranges", "nonnegative Unix timestamps", bool((RATINGS["timestamp"] >= 0).all()), int(RATINGS["timestamp"].min()), ">= 0")
    add_check("ranges", "timestamps parse as UTC", bool(TIMESTAMP_UTC.notna().all()), int(TIMESTAMP_UTC.isna().sum()), 0)
    add_check("ranges", "documented gender codes", set(USERS["gender"].unique()) <= EXPECTED_GENDERS, sorted(USERS["gender"].unique().tolist()), sorted(EXPECTED_GENDERS))
    add_check("ranges", "documented age codes", set(USERS["age"].unique()) <= EXPECTED_AGE_CODES, sorted(USERS["age"].unique().tolist()), sorted(EXPECTED_AGE_CODES))
    add_check("ranges", "documented occupation codes", set(USERS["occupation"].unique()) <= EXPECTED_OCCUPATION_CODES, sorted(USERS["occupation"].unique().tolist()), sorted(EXPECTED_OCCUPATION_CODES))
    add_check("ranges", "documented genre values", not UNKNOWN_GENRES, sorted(UNKNOWN_GENRES), [])

    users_missing_metadata = set(RATINGS["user_id"]) - set(USERS["user_id"])
    items_missing_metadata = set(RATINGS["item_id"]) - set(CANONICAL_ITEMS["item_id"])
    add_check("references", "all rated users have metadata", not users_missing_metadata, len(users_missing_metadata), 0)
    add_check("references", "all rated items have metadata", not items_missing_metadata, len(items_missing_metadata), 0)
    add_check(
        "coverage",
        "every metadata user has ratings",
        set(RATINGS["user_id"]) == set(USERS["user_id"]),
        int(RATINGS["user_id"].nunique()),
        int(USERS["user_id"].nunique()),
    )
    add_check("coverage", "at least 20 ratings per user", int(USER_ACTIVITY.min()) >= 20, int(USER_ACTIVITY.min()), ">= 20")

    empty_titles = int(CANONICAL_ITEMS["title"].str.strip().eq("").sum())
    empty_genres = int(CANONICAL_ITEMS["genres"].str.strip().eq("").sum())
    empty_zip_codes = int(USERS["zip_code"].str.strip().eq("").sum())
    missing_years = int(CANONICAL_ITEMS["release_year"].isna().sum())
    add_check("content", "nonempty titles", empty_titles == 0, empty_titles, 0)
    add_check("content", "nonempty genres", empty_genres == 0, empty_genres, 0)
    add_check("content", "nonempty ZIP codes", empty_zip_codes == 0, empty_zip_codes, 0)
    add_check("derived", "release year parsed", missing_years == 0, missing_years, 0, failure_status="WARN")

DATA_AUDIT_PASS = DATA_PARSED and not any(
    row["status"] == "FAIL" for row in AUDIT_CHECKS
)

show_records(
    AUDIT_CHECKS,
    ["category", "check", "status", "observed", "expected"],
)
display(Markdown("### Data-level audit: " + ("PASS" if DATA_AUDIT_PASS else "FAIL")))

,category,check,status,observed,expected
0,environment,required dependencies,PASS,[],[]
1,source,MovieLens-1M source found,PASS,"{'kind': 'zip', 'origin': 'input root', 'name'...",official source
2,source,required source members,PASS,"[ratings.dat, users.dat, movies.dat, README]","[ratings.dat, users.dat, movies.dat, README]"
3,source,README schema authority,PASS,True,True
4,source,official archive identity,PASS,{'archive_md5': 'c4d9eecfca2ab87c1945afe126590...,trusted archive or exact trusted member finger...
5,parsing,all canonical tables parsed,PASS,None,no error
6,release fingerprint,ratings row count,PASS,1000209,1000209
7,missingness,ratings missing values,PASS,0,0
8,release fingerprint,users row count,PASS,6040,6040
9,missingness,users missing values,PASS,0,0


### Data-level audit: PASS

## Descriptive Analysis

Summarize dataset scale, interaction sparsity, activity distributions, and
temporal behavior relevant to an item cold-start study. Activity counts are
audit summaries only and are not persisted as model features.

In [35]:
AUDIT_SUMMARY: dict[str, Any] = {
    "dataset": DATASET_NAME,
    "release": DATASET_RELEASE,
    "audit_schema_version": AUDIT_SCHEMA_VERSION,
    "source_found": DATA_SOURCE is not None,
    "data_parsed": DATA_PARSED,
    "data_audit_pass": DATA_AUDIT_PASS,
}
RATING_DISTRIBUTION = None
USER_ACTIVITY_SUMMARY = None
ITEM_ACTIVITY_SUMMARY = None

if DATA_PARSED:
    catalog_capacity = len(USERS) * len(CANONICAL_ITEMS)
    density = len(RATINGS) / catalog_capacity
    AUDIT_SUMMARY.update(
        {
            "ratings": int(len(RATINGS)),
            "users": int(len(USERS)),
            "catalog_items": int(len(CANONICAL_ITEMS)),
            "rated_items": int(RATINGS["item_id"].nunique()),
            "matrix_density": float(density),
            "matrix_sparsity": float(1 - density),
            "timestamp_min_utc": TIMESTAMP_UTC.min().isoformat(),
            "timestamp_max_utc": TIMESTAMP_UTC.max().isoformat(),
            "unparsed_release_years": int(CANONICAL_ITEMS["release_year"].isna().sum()),
        }
    )
    RATING_DISTRIBUTION = RATINGS["rating"].value_counts().sort_index()
    USER_ACTIVITY_SUMMARY = USER_ACTIVITY.describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
    ITEM_ACTIVITY_SUMMARY = ITEM_ACTIVITY.describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )

show_records([AUDIT_SUMMARY])
if DATA_PARSED:
    show_records(
        [
            {"distribution": "user interactions", **USER_ACTIVITY_SUMMARY.to_dict()},
            {"distribution": "item interactions", **ITEM_ACTIVITY_SUMMARY.to_dict()},
        ]
    )
    show_records(
        [
            {"rating": int(rating), "count": int(count)}
            for rating, count in RATING_DISTRIBUTION.items()
        ]
    )

,dataset,release,audit_schema_version,source_found,data_parsed,data_audit_pass,ratings,users,catalog_items,rated_items,matrix_density,matrix_sparsity,timestamp_min_utc,timestamp_max_utc,unparsed_release_years
0,MovieLens-1M,ml-1m,ml1m-audit-v2,True,True,True,1000209,6040,3883,3706,0.042647,0.957353,2000-04-25T23:05:32+00:00,2003-02-28T17:49:50+00:00,0


,distribution,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
0,user interactions,6040.0,165.597517,192.747029,20.0,44.0,96.0,208.0,400.0,556.0,906.66,2314.0
1,item interactions,3706.0,269.889099,384.047838,1.0,33.0,123.5,350.0,729.5,1051.5,1784.90,3428.0


,rating,count
0,1,56174
1,2,107557
2,3,261197
3,4,348971
4,5,226310


In [36]:
if DATA_PARSED and pyplot is not None:
    figure, axes = pyplot.subplots(1, 3, figsize=(16, 4))

    axes[0].bar(RATING_DISTRIBUTION.index.astype(str), RATING_DISTRIBUTION.values)
    axes[0].set_title("Rating distribution")
    axes[0].set_xlabel("Rating")
    axes[0].set_ylabel("Interactions")

    axes[1].hist(ITEM_ACTIVITY.values, bins=50)
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_title("Interactions per item")
    axes[1].set_xlabel("Interaction count, log scale")

    monthly_activity = pandas.Series(1, index=TIMESTAMP_UTC).resample("MS").sum()
    axes[2].plot(monthly_activity.index, monthly_activity.values)
    axes[2].set_title("Interactions over time")
    axes[2].set_xlabel("Month")
    axes[2].tick_params(axis="x", rotation=30)

    figure.tight_layout()
    pyplot.show()
else:
    print("Plots skipped: parsed data and Matplotlib are both required.")

Plots skipped: parsed data and Matplotlib are both required.


## EmerG Feature Inventory

Identify only documented side information that can support EmerG and DGD.
LightGCN remains interaction-only. No vocabulary, contiguous ID map, target,
or model tensor is fitted in this notebook.

In [37]:
def profile(table: Any, column: str) -> tuple[int | None, int | None]:
    if table is None:
        return None, None
    return int(table[column].nunique(dropna=True)), int(table[column].isna().sum())


def feature_row(
    feature: str,
    source: str,
    side: str,
    table: Any,
    raw_or_derived: str,
    lightgcn_role: str,
    emerg_dgd_role: str,
    downstream_action: str,
    leakage_control: str,
) -> dict[str, Any]:
    cardinality, missing = profile(table, feature)
    return {
        "feature": feature,
        "source": source,
        "side": side,
        "raw_or_derived": raw_or_derived,
        "cardinality": cardinality,
        "missing": missing,
        "lightgcn_role": lightgcn_role,
        "emerg_dgd_role": emerg_dgd_role,
        "downstream_action": downstream_action,
        "leakage_control": leakage_control,
    }


FEATURE_INVENTORY = [
    feature_row("user_id", "ratings.dat/users.dat", "user", USERS, "raw", "graph identity", "categorical identity", "map after split", "fit mapping on protocol-visible IDs"),
    feature_row("gender", "users.dat", "user", USERS, "raw", "none", "categorical feature", "encode after split", "fit encoder on training-visible data"),
    feature_row("age", "users.dat", "user", USERS, "raw code", "none", "categorical feature", "encode documented code", "fit encoder on training-visible data"),
    feature_row("occupation", "users.dat", "user", USERS, "raw code", "none", "categorical feature", "encode documented code", "fit encoder on training-visible data"),
    feature_row("zip_code", "users.dat", "user", USERS, "raw string", "none", "categorical feature", "encode after split", "fit vocabulary on training-visible data"),
    feature_row("item_id", "ratings.dat/movies.dat", "item", CANONICAL_ITEMS, "raw", "graph identity", "categorical identity", "map after split", "preserve cold-item membership"),
    feature_row("title", "movies.dat", "item", CANONICAL_ITEMS, "raw string", "none", "sequence feature", "tokenize after split", "fit vocabulary on training-visible data"),
    feature_row("release_year", "movies.dat:title", "item", CANONICAL_ITEMS, "derived", "none", "categorical feature", "encode after split", "deterministic title-only derivation"),
    feature_row("genres", "movies.dat", "item", CANONICAL_ITEMS, "raw pipe list", "none", "multi-value feature", "encode after split", "fit vocabulary on training-visible data"),
    feature_row("rating", "ratings.dat", "interaction", RATINGS, "raw", "edge policy deferred", "target source", "derive label in notebook 02", "never expose held-out targets"),
    feature_row("timestamp", "ratings.dat", "interaction", RATINGS, "raw", "protocol ordering", "not a model feature", "use for chronological phases", "never encode future time as a feature"),
]

show_records(
    FEATURE_INVENTORY,
    [
        "feature",
        "source",
        "side",
        "raw_or_derived",
        "cardinality",
        "missing",
        "lightgcn_role",
        "emerg_dgd_role",
        "downstream_action",
        "leakage_control",
    ],
)

,feature,source,side,raw_or_derived,cardinality,missing,lightgcn_role,emerg_dgd_role,downstream_action,leakage_control
0,user_id,ratings.dat/users.dat,user,raw,6040,0,graph identity,categorical identity,map after split,fit mapping on protocol-visible IDs
1,gender,users.dat,user,raw,2,0,none,categorical feature,encode after split,fit encoder on training-visible data
2,age,users.dat,user,raw code,7,0,none,categorical feature,encode documented code,fit encoder on training-visible data
3,occupation,users.dat,user,raw code,21,0,none,categorical feature,encode documented code,fit encoder on training-visible data
4,zip_code,users.dat,user,raw string,3439,0,none,categorical feature,encode after split,fit vocabulary on training-visible data
5,item_id,ratings.dat/movies.dat,item,raw,3883,0,graph identity,categorical identity,map after split,preserve cold-item membership
6,title,movies.dat,item,raw string,3883,0,none,sequence feature,tokenize after split,fit vocabulary on training-visible data
7,release_year,movies.dat:title,item,derived,81,0,none,categorical feature,encode after split,deterministic title-only derivation
8,genres,movies.dat,item,raw pipe list,301,0,none,multi-value feature,encode after split,fit vocabulary on training-visible data
9,rating,ratings.dat,interaction,raw,5,0,edge policy deferred,target source,derive label in notebook 02,never expose held-out targets


## Validated Handoff to Notebook 02

A passing audit writes deterministic UTF-8 CSV tables and machine-readable
metadata under `processed/ml-1m/audit-v2`. The original rating and timestamp
values are preserved. The manifest is written last so notebook 02 never treats
a partial export as valid.

In [38]:
def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return json_ready(value.item())
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)


def write_text_atomic(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_dataframe_atomic(path: Path, table: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, encoding="utf-8", lineterminator="\n")
    temporary.replace(path)


def relative_artifact_path(path: Path) -> str:
    try:
        return str(path.relative_to(ARTIFACT_ROOT))
    except ValueError:
        return str(path)


def render_audit_summary(manifest: dict[str, Any]) -> str:
    failures = [row for row in AUDIT_CHECKS if row["status"] == "FAIL"]
    warnings = [row for row in AUDIT_CHECKS if row["status"] == "WARN"]
    lines = [
        "# MovieLens-1M Data Audit",
        "",
        f"- Status: {manifest['audit_status']}",
        f"- Source: {manifest['source']['kind']} ({manifest['source']['origin']})",
        f"- Ratings: {AUDIT_SUMMARY.get('ratings')}",
        f"- Users: {AUDIT_SUMMARY.get('users')}",
        f"- Catalog items: {AUDIT_SUMMARY.get('catalog_items')}",
        f"- Failures: {len(failures)}",
        f"- Warnings: {len(warnings)}",
        "",
        "Notebook 02 may proceed only when this status is PASS and every artifact hash matches.",
    ]
    return "\n".join(lines) + "\n"

In [39]:
WRITE_ARTIFACTS = enabled("COLDSTART_WRITE_ARTIFACTS", default=True)
EXPORT_STATUS = "BLOCKED"
EXPORT_ERROR = None
OUTPUT_ARTIFACTS: dict[str, dict[str, Any]] = {}
MANIFEST_PATH = AUDIT_OUTPUT_ROOT / "manifest.json"
BUNDLE_ID = None
GENERATION_ROOT = None
STAGING_ROOT = None

HANDOFF_PATH_CONTRACT = {
    "manifest_path_from_audit_root": "processed/ml-1m/audit-v2/manifest.json",
    "artifact_paths": "relative to COLDSTART_AUDIT_ROOT when consumed downstream",
    "local_audit_root": "COLDSTART_ARTIFACT_ROOT",
    "kaggle_downstream_input": "set COLDSTART_AUDIT_ROOT to the mounted notebook-01 output",
    "downstream_outputs": "keep COLDSTART_ARTIFACT_ROOT on writable local or Kaggle working storage",
}

if DATA_AUDIT_PASS and WRITE_ARTIFACTS and pandas is not None:
    try:
        BUNDLE_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:12]
        STAGING_ROOT = AUDIT_OUTPUT_ROOT / f".staging-{BUNDLE_ID}"
        GENERATION_ROOT = AUDIT_OUTPUT_ROOT / "generations" / BUNDLE_ID
        STAGING_ROOT.mkdir(parents=True, exist_ok=False)

        tables = {
            "interactions": ("interactions.csv", RATINGS),
            "users": ("users.csv", USERS),
            "items": ("items.csv", CANONICAL_ITEMS),
            "feature_inventory": (
                "feature_inventory.csv",
                pandas.DataFrame(FEATURE_INVENTORY),
            ),
            "audit_checks": (
                "audit_checks.csv",
                pandas.DataFrame(AUDIT_CHECKS),
            ),
            "source_inventory": (
                "source_inventory.csv",
                pandas.DataFrame(SOURCE_INVENTORY),
            ),
        }

        for name, (file_name, table) in tables.items():
            staging_path = STAGING_ROOT / file_name
            published_path = GENERATION_ROOT / file_name
            write_dataframe_atomic(staging_path, table)
            OUTPUT_ARTIFACTS[name] = {
                "path": relative_artifact_path(published_path),
                "sha256": file_digest(staging_path),
                "rows": int(len(table)),
            }

        summary_json_path = STAGING_ROOT / "audit_summary.json"
        write_text_atomic(
            summary_json_path,
            json.dumps(json_ready(AUDIT_SUMMARY), indent=2, sort_keys=True) + "\n",
        )
        OUTPUT_ARTIFACTS["audit_summary_json"] = {
            "path": relative_artifact_path(GENERATION_ROOT / "audit_summary.json"),
            "sha256": file_digest(summary_json_path),
        }

        source_payload = {
            "kind": DATA_SOURCE.kind,
            "origin": DATA_SOURCE.origin,
            "name": DATA_SOURCE.location.name,
            "archive_md5": ARCHIVE_MD5,
            "archive_sha256": ARCHIVE_SHA256,
            "members": SOURCE_INVENTORY,
        }
        manifest = {
            "dataset": DATASET_NAME,
            "release": DATASET_RELEASE,
            "audit_schema_version": AUDIT_SCHEMA_VERSION,
            "audit_status": "PASS",
            "bundle_id": BUNDLE_ID,
            "bundle_manifest": relative_artifact_path(GENERATION_ROOT / "manifest.json"),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "upstream_notebook": "00_environment_and_reproducibility.ipynb",
            "downstream_notebook": "02_cold_start_protocol.ipynb",
            "path_contract": HANDOFF_PATH_CONTRACT,
            "distribution_policy": "private research artifact; respect GroupLens no-redistribution terms",
            "official_sources": {
                "archive_url": OFFICIAL_ARCHIVE_URL,
                "readme_url": OFFICIAL_README_URL,
                "archive_md5": OFFICIAL_ARCHIVE_MD5,
                "trusted_archive_sha256": TRUSTED_ARCHIVE_SHA256,
                "trusted_member_fingerprints": TRUSTED_MEMBER_FINGERPRINTS,
            },
            "source": source_payload,
            "source_schemas": SOURCE_SCHEMAS,
            "canonical_output_schemas": CANONICAL_OUTPUT_SCHEMAS,
            "canonical_item_derived_columns": {
                "release_year": "four trailing digits in the documented title",
                "release_year_parse_status": "parsed or missing",
            },
            "summary": AUDIT_SUMMARY,
            "checks": AUDIT_CHECKS,
            "artifacts": OUTPUT_ARTIFACTS,
            "parser": {
                "python": platform.python_version(),
                "pandas": metadata.version("pandas"),
            },
            "deferred_to_notebook_02": [
                "binary label",
                "item-frequency cohorts",
                "train-validation-test split",
                "Cold and Warm-Up A/B/C phases",
                "candidate policy",
                "contiguous ID mappings",
                "feature encoders and vocabularies",
            ],
        }

        summary_markdown_path = STAGING_ROOT / "audit_summary.md"
        write_text_atomic(summary_markdown_path, render_audit_summary(manifest))
        OUTPUT_ARTIFACTS["audit_summary_markdown"] = {
            "path": relative_artifact_path(GENERATION_ROOT / "audit_summary.md"),
            "sha256": file_digest(summary_markdown_path),
        }
        manifest["artifacts"] = OUTPUT_ARTIFACTS

        bundle_manifest_path = STAGING_ROOT / "manifest.json"
        write_text_atomic(
            bundle_manifest_path,
            json.dumps(json_ready(manifest), indent=2, sort_keys=True) + "\n",
        )

        GENERATION_ROOT.parent.mkdir(parents=True, exist_ok=True)
        STAGING_ROOT.replace(GENERATION_ROOT)
        STAGING_ROOT = None

        # Publish the complete immutable generation with one atomic pointer swap.
        write_text_atomic(
            MANIFEST_PATH,
            json.dumps(json_ready(manifest), indent=2, sort_keys=True) + "\n",
        )
        EXPORT_STATUS = "PASS"
    except Exception as error:
        EXPORT_STATUS = "FAIL"
        EXPORT_ERROR = repr(error)
        if STAGING_ROOT is not None and STAGING_ROOT.exists():
            shutil.rmtree(STAGING_ROOT, ignore_errors=True)
elif DATA_AUDIT_PASS and not WRITE_ARTIFACTS:
    EXPORT_STATUS = "SKIPPED"

show_records(
    [
        {
            "write_artifacts": WRITE_ARTIFACTS,
            "export_status": EXPORT_STATUS,
            "export_error": EXPORT_ERROR,
            "manifest_path": str(MANIFEST_PATH),
            "manifest_exists": MANIFEST_PATH.is_file(),
            "artifacts": OUTPUT_ARTIFACTS,
        }
    ]
)

,write_artifacts,export_status,export_error,manifest_path,manifest_exists,artifacts
0,True,PASS,None,/workspace/HungPH/coldstart-recsys/.notebook/a...,True,{'interactions': {'path': 'processed/ml-1m/aud...


## Audit Decision

Notebook execution success is separate from data readiness. Missing local
dependencies or an unattached archive should produce a complete, actionable
failure report rather than a cell exception.

In [40]:
HANDOFF_READY = DATA_AUDIT_PASS and EXPORT_STATUS == "PASS" and MANIFEST_PATH.is_file()
FEATURE_PROFILE_COMPLETE = DATA_PARSED and all(
    row["cardinality"] is not None and row["missing"] is not None
    for row in FEATURE_INVENTORY
)
if DATA_PARSED:
    parsing_detail = "ratings, users, and items parsed"
elif PARSE_ERROR:
    parsing_detail = PARSE_ERROR
else:
    parsing_detail = "parsing not attempted because a trusted source or required dependency is unavailable"

FINAL_VALIDATION = [
    {
        "requirement": "notebook 00 path contract applied",
        "status": "PASS",
        "detail": str(ARTIFACT_ROOT),
    },
    {
        "requirement": "required audit dependencies",
        "status": "PASS" if not MISSING_REQUIRED else "FAIL",
        "detail": MISSING_REQUIRED or "available",
    },
    {
        "requirement": "official MovieLens-1M source",
        "status": "PASS" if SOURCE_IDENTITY_VALID else "FAIL",
        "detail": SOURCE_REFERENCE or SOURCE_MESSAGES,
    },
    {
        "requirement": "official README contract",
        "status": "PASS" if README_AUTHORITY_VALID else "FAIL",
        "detail": OFFICIAL_README_URL,
    },
    {
        "requirement": "lossless canonical parsing",
        "status": "PASS" if DATA_PARSED else "FAIL",
        "detail": parsing_detail,
    },
    {
        "requirement": "integrity audit",
        "status": "PASS" if DATA_AUDIT_PASS else "FAIL",
        "detail": f"{len(AUDIT_CHECKS)} checks",
    },
    {
        "requirement": "descriptive audit",
        "status": "PASS" if DATA_PARSED else "FAIL",
        "detail": "scale, sparsity, temporal, rating, and activity summaries",
    },
    {
        "requirement": "EmerG/DGD feature inventory",
        "status": "PASS" if FEATURE_PROFILE_COMPLETE else "FAIL",
        "detail": f"{len(FEATURE_INVENTORY)} audited fields",
    },
    {
        "requirement": "notebook 02 handoff manifest",
        "status": "PASS" if HANDOFF_READY else "FAIL",
        "detail": EXPORT_ERROR or str(MANIFEST_PATH),
    },
]

show_records(FINAL_VALIDATION, ["requirement", "status", "detail"])
display(Markdown("### Notebook 01 data audit: " + ("PASS" if DATA_AUDIT_PASS else "FAIL")))
display(Markdown("### Notebook 02 handoff: " + ("READY" if HANDOFF_READY else "BLOCKED")))

,requirement,status,detail
0,notebook 00 path contract applied,PASS,/workspace/HungPH/coldstart-recsys/.notebook/a...
1,required audit dependencies,PASS,available
2,official MovieLens-1M source,PASS,"{'kind': 'zip', 'origin': 'input root', 'name'..."
3,official README contract,PASS,https://files.grouplens.org/datasets/movielens...
4,lossless canonical parsing,PASS,"ratings, users, and items parsed"
5,integrity audit,PASS,32 checks
6,descriptive audit,PASS,"scale, sparsity, temporal, rating, and activit..."
7,EmerG/DGD feature inventory,PASS,11 audited fields
8,notebook 02 handoff manifest,PASS,/workspace/HungPH/coldstart-recsys/.notebook/a...


### Notebook 01 data audit: PASS

### Notebook 02 handoff: READY

## Continue to Notebook 02

Notebook 02 should:

1. Locate `processed/ml-1m/audit-v2/manifest.json` under `COLDSTART_AUDIT_ROOT`.
2. Require `audit_status == "PASS"` and verify every listed SHA-256 hash.
3. Load canonical CSVs with the exact dtypes declared by this audit.
4. Confirm raw IDs, ratings, and timestamps still match the manifest.
5. Derive the proposal's binary feedback label and item-frequency policy.
6. Construct item-level train/validation/test and Cold/Warm-Up A/B/C phases.
7. Fit all ID maps and feature vocabularies only after protocol boundaries exist.

Locally, `COLDSTART_AUDIT_ROOT` can equal `COLDSTART_ARTIFACT_ROOT`. On Kaggle,
set it to the read-only mounted notebook-01 output while keeping
`COLDSTART_ARTIFACT_ROOT` on writable `/kaggle/working` storage for protocol
outputs.

For local execution, place `ml-1m.zip` under the configured input root, set
`MOVIELENS_1M_PATH`, or explicitly opt into download. For Kaggle, attach the
official archive as a private dataset because internet remains disabled, then
expose notebook 01 outputs to notebook 02 as a private dataset or kernel output.